# Conversione UniMiB-SHAR Complete in CSV

Questo notebook legge `ml_accelerometers/UniMib_SHAR_complete/` in sola lettura e crea copie CSV in `fall_detection_ml/data/UniMiBComplete/`.

La conversione conserva:
- una rappresentazione lossless a sei colonne (`x`, `y`, `z`, timestamp e magnitudine);
- una rappresentazione ML a tre colonne compatibile con il loader esistente;
- label, soggetto, prova, tasca e dati demografici;
- opzionalmente, le 11.771 finestre ufficiali da 151 campioni in formato temporale.

Non modifica i file sorgente e non modifica le procedure esistenti del progetto.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

PROJECT_DIR = Path.cwd().resolve()
SOURCE_DIR = PROJECT_DIR.parent / "ml_accelerometers" / "UniMib_SHAR_complete"
MAT_DATA_DIR = SOURCE_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "data" / "UniMiBComplete"

RECORDINGS_ML_DIR = OUTPUT_DIR / "recordings_ml"
RECORDINGS_FULL_DIR = OUTPUT_DIR / "recordings_full"

# Le finestre ufficiali sono circa 1,78 milioni di righe in formato long.
EXPORT_WINDOWS_151 = True
FLOAT_FORMAT = "%.17g"

assert SOURCE_DIR.is_dir(), f"Dataset sorgente non trovato: {SOURCE_DIR}"
assert PROJECT_DIR.name == "fall_detection_ml", "Avvia il notebook da fall_detection_ml/"

print("Sorgente:", SOURCE_DIR)
print("Destinazione:", OUTPUT_DIR)

## Mapping delle classi
Il progetto esistente usa un ordine diverso da `acc_names.mat`. Il mapping seguente evita dipendenze accidentali dall'ordine dei campi MATLAB.

In [ ]:
PROJECT_CLASSES = [
    "Walking", "Running", "GoingUpS", "GoingDownS", "Jumping",
    "SittingDown", "StandingUpFS", "LyingDownFS", "StandingUpFL",
    "FallingForw", "FallingBack", "FallingRight", "FallingLeft",
    "Syncope", "FallingBackSC", "FallingWithPS", "HittingObstacle",
]

SOURCE_CLASSES = [
    "StandingUpFS", "StandingUpFL", "Walking", "Running",
    "GoingUpS", "Jumping", "GoingDownS", "LyingDownFS",
    "SittingDown", "FallingForw", "FallingRight",
    "FallingBack", "HittingObstacle", "FallingWithPS",
    "FallingBackSC", "Syncope", "FallingLeft",
]

DISPLAY_NAMES = [
    "Standing up from sitting", "Standing up from laying",
    "Walking", "Running", "Going upstairs", "Jumping",
    "Going downstairs", "Lying down from standing", "Sitting down",
    "Generic falling forward", "Falling rightward",
    "Generic falling backward", "Hitting an obstacle in the fall",
    "Falling with protection strategies",
    "Falling backward-sitting-chair", "Syncope", "Falling leftward",
]

PROJECT_LABEL = {name: i for i, name in enumerate(PROJECT_CLASSES)}
SOURCE_LABEL = {name: i + 1 for i, name in enumerate(SOURCE_CLASSES)}
DISPLAY_BY_SOURCE = dict(zip(SOURCE_CLASSES, DISPLAY_NAMES))

class_map = pd.DataFrame({
    "activity_name": PROJECT_CLASSES,
    "project_label": [PROJECT_LABEL[name] for name in PROJECT_CLASSES],
    "source_label": [SOURCE_LABEL[name] for name in PROJECT_CLASSES],
    "display_name": [DISPLAY_BY_SOURCE[name] for name in PROJECT_CLASSES],
    "coarse_label": [0 if PROJECT_LABEL[name] < 9 else 1 for name in PROJECT_CLASSES],
    "coarse_name": ["adl" if PROJECT_LABEL[name] < 9 else "fall" for name in PROJECT_CLASSES],
})
class_map

## Controlli preliminari del file MATLAB
La conversione fallisce subito se cambiano struttura, numero di soggetti, attività o prove.

In [ ]:
full_data = loadmat(
    MAT_DATA_DIR / "full_data.mat",
    struct_as_record=False,
    squeeze_me=False,
)["full_data"]

assert full_data.shape == (30, 5), full_data.shape
first_subject = full_data[0, 0][0, 0]
assert set(first_subject._fieldnames) == set(PROJECT_CLASSES)

trial_counts = {}
for subject_index in range(30):
    subject = full_data[subject_index, 0][0, 0]
    for activity_name in PROJECT_CLASSES:
        expected_trials = 2 if PROJECT_LABEL[activity_name] < 9 else 6
        trials = getattr(subject, activity_name)
        assert trials.size == expected_trials, (subject_index, activity_name, trials.size)
        for trial in trials.flat:
            assert trial.ndim == 2 and trial.shape[0] == 6
            assert np.issubdtype(trial.dtype, np.floating)
            trial_counts[activity_name] = trial_counts.get(activity_name, 0) + 1

assert sum(trial_counts.values()) == 1980
print("Soggetti: 30")
print("Registrazioni attese e trovate: 1.980")
print("Attività:", first_subject._fieldnames)

## Conversione delle 1.980 registrazioni
Per il file con soli tre assi viene rimossa esclusivamente una coda strutturale in cui X/Y/Z e magnitudine sono tutti `NaN`. Le righe temporali valide non vengono ricampionate o trasformate. La copia `recordings_full` conserva anche i NaN originali.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECORDINGS_ML_DIR.mkdir(exist_ok=True)
RECORDINGS_FULL_DIR.mkdir(exist_ok=True)

record_rows = []
subject_rows = []
label_rows = []

for subject_index in range(30):
    subject_struct = full_data[subject_index, 0][0, 0]
    gender = str(full_data[subject_index, 1][0]).strip()
    age = int(full_data[subject_index, 2][0, 0])
    height = int(full_data[subject_index, 3][0, 0])
    weight = int(full_data[subject_index, 4][0, 0])

    subject_rows.append({
        "subject_id": subject_index,
        "source_subject_id": subject_index + 1,
        "gender": gender,
        "age": age,
        "height_source_units": height,
        "weight_source_units": weight,
    })

    file_index = 0
    for activity_name in PROJECT_CLASSES:
        project_label = PROJECT_LABEL[activity_name]
        coarse_label = 0 if project_label < 9 else 1
        coarse_name = "adl" if coarse_label == 0 else "fall"

        for trial_index, raw_matrix in enumerate(getattr(subject_struct, activity_name).flat, start=1):
            record_id = f"subject_{subject_index}_file_{file_index}.csv"
            full_values = raw_matrix.T

            # Righe con assi e magnitudine mancanti, ma timestamp ancora presenti.
            signal_missing = np.isnan(full_values[:, [0, 1, 2, 5]]).all(axis=1)
            valid_signal = ~signal_missing
            if np.any(valid_signal):
                last_valid = int(np.flatnonzero(valid_signal)[-1])
                has_internal_gap = bool(np.any(signal_missing[:last_valid + 1]))
            else:
                raise ValueError(f"Registrazione senza segnale valido: {record_id}")
            assert not has_internal_gap, f"NaN interni non previsti: {record_id}"

            ml_values = full_values[:last_valid + 1, :3]
            if project_label < 9:
                assert trial_index in (1, 2)
                pocket = "right" if trial_index == 1 else "left"
            else:
                assert trial_index in range(1, 7)
                pocket = "right" if trial_index <= 3 else "left"

            np.savetxt(
                RECORDINGS_ML_DIR / record_id,
                ml_values,
                delimiter=",",
                header="x,y,z",
                comments="# ",
                fmt=FLOAT_FORMAT,
            )
            np.savetxt(
                RECORDINGS_FULL_DIR / record_id,
                full_values,
                delimiter=",",
                header="x,y,z,time_ns,time_s,magnitude",
                comments="# ",
                fmt=FLOAT_FORMAT,
            )

            record_rows.append({
                "record_id": record_id,
                "subject_id": subject_index,
                "source_subject_id": subject_index + 1,
                "trial_id": trial_index,
                "activity_name": activity_name,
                "label": project_label,
                "source_label": SOURCE_LABEL[activity_name],
                "coarse_label": coarse_label,
                "coarse_name": coarse_name,
                "pocket": pocket,
                "gender": gender,
                "age": age,
                "height_source_units": height,
                "weight_source_units": weight,
                "n_samples": int(ml_values.shape[0]),
                "source_n_rows": int(full_values.shape[0]),
                "trailing_missing_rows": int(full_values.shape[0] - ml_values.shape[0]),
                "source_file": "full_data.mat",
                "source_field": activity_name,
            })
            label_rows.append({
                "record_id": record_id,
                "label": project_label,
                "label_name": activity_name,
            })
            file_index += 1

    assert file_index == 66, (subject_index, file_index)

records_meta = pd.DataFrame(record_rows)
subjects = pd.DataFrame(subject_rows)
labels = pd.DataFrame(label_rows)

assert len(records_meta) == len(labels) == 1980
assert records_meta["record_id"].is_unique

records_meta.to_csv(OUTPUT_DIR / "recordings_meta.csv", index=False)
subjects.to_csv(OUTPUT_DIR / "subjects.csv", index=False)
labels.to_csv(OUTPUT_DIR / "labels.csv", index=False)
class_map.to_csv(OUTPUT_DIR / "class_map.csv", index=False)

# Compatibile con np.loadtxt usato dal progetto corrente.
np.savetxt(
    OUTPUT_DIR / "labels_only.csv",
    labels["label"].to_numpy(dtype=int),
    fmt="%d",
    header="True_label",
    comments="# ",
)

print("CSV ML scritti:", len(list(RECORDINGS_ML_DIR.glob("*.csv"))))
print("CSV completi scritti:", len(list(RECORDINGS_FULL_DIR.glob("*.csv"))))
print(records_meta[["n_samples", "source_n_rows", "trailing_missing_rows"]].describe())

## Verifica di corrispondenza
Per ogni CSV ML viene controllata la conversione contro il MAT. I CSV completi possono avere i NaN strutturali della sorgente e vengono verificati con confronto elemento per elemento usando `equal_nan=True`.

In [ ]:
checked_ml = 0
checked_full = 0

for subject_index in range(30):
    subject_struct = full_data[subject_index, 0][0, 0]
    file_index = 0
    for activity_name in PROJECT_CLASSES:
        for raw_matrix in getattr(subject_struct, activity_name).flat:
            record_id = f"subject_{subject_index}_file_{file_index}.csv"
            expected_full = raw_matrix.T
            expected_valid = ~np.isnan(expected_full[:, [0, 1, 2, 5]]).all(axis=1)
            n_valid = int(np.flatnonzero(expected_valid)[-1] + 1)

            actual_ml = np.loadtxt(RECORDINGS_ML_DIR / record_id, delimiter=",")
            assert np.array_equal(actual_ml, expected_full[:n_valid, :3])

            actual_full = np.loadtxt(RECORDINGS_FULL_DIR / record_id, delimiter=",")
            assert np.array_equal(actual_full, expected_full, equal_nan=True)

            checked_ml += 1
            checked_full += 1
            file_index += 1

print(f"Verificati {checked_ml} CSV ML e {checked_full} CSV completi.")

## Esportazione opzionale delle finestre ufficiali da 151 campioni
`acc_data.mat` è originariamente una matrice `11771 × 453`, con i blocchi ordinati X, Y, Z. Qui viene ricostruita una tabella long-form: una riga per campione, tre assi separati. Non si eseguono normalizzazione, filtraggio o ricampionamento.

In [ ]:
if EXPORT_WINDOWS_151:
    acc_data = loadmat(MAT_DATA_DIR / "acc_data.mat")["acc_data"]
    acc_labels = loadmat(MAT_DATA_DIR / "acc_labels.mat")["acc_labels"]
    assert acc_data.shape == (11771, 453)
    assert acc_labels.shape == (11771, 3)

    windows_dir = OUTPUT_DIR / "windows151"
    windows_dir.mkdir(exist_ok=True)
    windows_csv = windows_dir / "acc_windows.csv"
    windows_meta_csv = windows_dir / "acc_windows_meta.csv"

    with windows_meta_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "window_id", "source_label", "label", "label_name",
            "coarse_label", "coarse_name", "source_subject_id",
            "subject_id", "trial_id", "n_samples",
        ])
        for window_index, (source_label, source_subject_id, trial_id) in enumerate(acc_labels):
            source_name = SOURCE_CLASSES[int(source_label) - 1]
            project_label = PROJECT_LABEL[source_name]
            writer.writerow([
                window_index, int(source_label), project_label, source_name,
                0 if project_label < 9 else 1,
                "adl" if project_label < 9 else "fall",
                int(source_subject_id), int(source_subject_id) - 1,
                int(trial_id), 151,
            ])

    with windows_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["window_id", "sample_index", "x", "y", "z"])
        for window_index, flat_window in enumerate(acc_data):
            window = flat_window.reshape(3, 151).T
            for sample_index, (x, y, z) in enumerate(window):
                writer.writerow([window_index, sample_index, float(x), float(y), float(z)])

    print("Finestre esportate:", len(acc_data))
    print("CSV finestre:", windows_csv)
else:
    print("Esportazione finestre disattivata.")

## Manifest e riepilogo
Il manifest registra forma, origine e scelte di conversione. Non contiene né normalizzazioni né trasformazioni numeriche.

In [ ]:
manifest = {
    "source": str(SOURCE_DIR),
    "output": str(OUTPUT_DIR),
    "source_format": "MATLAB v5",
    "subjects": int(len(subjects)),
    "recordings": int(len(records_meta)),
    "activities": int(len(PROJECT_CLASSES)),
    "ml_csv_columns": ["x", "y", "z"],
    "full_csv_columns": ["x", "y", "z", "time_ns", "time_s", "magnitude"],
    "trimming": "remove only trailing rows with NaN in x/y/z/magnitude",
    "normalization": False,
    "filtering": False,
    "resampling": False,
    "windows_151_exported": bool(EXPORT_WINDOWS_151),
    "windows_151_count": 11771 if EXPORT_WINDOWS_151 else 0,
    "source_units_documented": False,
    "inferred_nominal_sample_rate_hz": 50,
    "sample_rate_is_officially_documented": False,
}

with (OUTPUT_DIR / "conversion_manifest.json").open("w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("Conversione terminata.")